# Stage Three: Model Construction, Testing and Selection

This notebook is a record of the code for Stage Three. What each result showed, and the reasoning behind each decision, are set out in `md/03_modelling_writeup.md`.

---

## 1. Load the feature set

This opens the database and reads the finished feature and value table into Python, then prints its shape and first few rows.

No data preparation happens here. Every measure was worked out in SQL and arrives ready to use. That is the intended split of work in this project: the database does the transformation, and the modelling step only consumes the result.

In [13]:
import duckdb
import pandas as pd

con = duckdb.connect("../telco.duckdb")

df = con.execute("SELECT * FROM vw_customer_value_segments").df()

print(df.shape)
df.head()

(7043, 18)


,customer_id,tenure_months,monthly_charges,total_charges,contract,internet_service,churn_value,tenure_band,has_internet,has_phone,addon_count,contract_risk,automatic_payment,spend_band,expected_remaining_months,historic_margin,clv,value_decile
0,7569-NMZYQ,72,118.75,8672.45,Two year,Fiber optic,0,Loyal,1,1,6,1,1,High,36,2601.74,1282.50,1
1,8984-HPEMB,71,118.65,8477.60,Two year,Fiber optic,0,Loyal,1,1,6,1,0,High,36,2543.28,1281.42,1
2,5989-AXPUC,68,118.60,7990.05,Two year,Fiber optic,0,Loyal,1,1,6,1,0,High,36,2397.02,1280.88,1
3,9924-JPRMC,72,118.20,8547.15,Two year,Fiber optic,0,Loyal,1,1,6,1,0,High,36,2564.15,1276.56,1
4,3810-DVDQQ,72,117.60,8308.90,Two year,Fiber optic,0,Loyal,1,1,6,1,1,High,36,2492.67,1270.08,1


## 2. Select the measures the model is allowed to use

This picks the fields the model may learn from, converts the one text field into numbers, and prints the result.

Not every column belongs in the model. Four groups are left out. `customer_id` labels a customer but says nothing about them. `churn_value` is the outcome being predicted and cannot also be an input. The customer value fields, `clv`, `historic_margin`, `expected_remaining_months` and `value_decile`, are left out so that how likely a customer is to leave and how much they are worth stay two separate questions, brought together only at Stage Four. The banded fields are left out because they repeat information already present: `tenure_band` repeats `tenure_months`, `spend_band` repeats `monthly_charges`, `contract` repeats `contract_risk` in words, and `has_internet` sits inside `internet_service`.

`internet_service` holds text, which a model cannot read, so `pd.get_dummies` turns it into numeric columns. `drop_first=True` removes one of the three, because once two are known the third is implied, and keeping all three would put the same information in twice.

In [14]:
target = "churn_value"

features = [
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "has_phone",
    "addon_count",
    "contract_risk",
    "automatic_payment",
    "internet_service",
]

X = pd.get_dummies(df[features], columns=["internet_service"], drop_first=True)
y = df[target]

print(X.shape)
X.head()

(7043, 9)


,tenure_months,monthly_charges,total_charges,has_phone,addon_count,contract_risk,automatic_payment,internet_service_Fiber optic,internet_service_No
0,72,118.75,8672.45,1,6,1,1,True,False
1,71,118.65,8477.60,1,6,1,0,True,False
2,68,118.60,7990.05,1,6,1,0,True,False
3,72,118.20,8547.15,1,6,1,0,True,False
4,72,117.60,8308.90,1,6,1,1,True,False


## 3. Separate the data into training and test sets

This splits the customers into two groups. The model learns from one and is measured on the other, which it never sees while training.

The split is what makes the measurement mean anything. A model measured on the same records it learned from can simply repeat what it memorised, and would report a level of performance it could not reach on new customers.

Three settings are used. `test_size=0.25` holds back a quarter of customers for measurement. `stratify=y` makes both groups hold the same share of departed customers as the full dataset, so a random split cannot hand the test set an unrepresentative share. `random_state=42` fixes the split so it is the same every time the notebook runs, which lets anyone re-running the work get the same numbers. The particular value is arbitrary.

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(5282, 9) (1761, 9)
0.2654297614539947 0.26519023282226006


## 4. Train the logistic regression

This builds the model as a pipeline of two steps, scaling followed by the regression itself, and fits it to the training group.

`StandardScaler` puts all the measures on a common scale. They are recorded in very different units, with total charges running into the thousands while the count of extra services runs from zero to six. Without scaling, the model would treat the larger numbers as more important purely because of their size.

`Pipeline` joins the scaling and the model into one object. This is about correctness rather than convenience: the scaler works out its adjustments from the training group only and then applies them to the test group. Doing the two steps separately makes it easy to scale everything together by mistake, which would let the test group influence training.

`class_weight="balanced"` deals with the imbalance in the data. Only about a quarter of customers left, so a model aiming at overall accuracy could score well by predicting that nobody ever leaves. This setting tells the model to treat a missed departure as the more serious mistake.

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

logreg.fit(X_train, y_train)
print("trained")

trained


## 5. Measure performance on the held-back customers

This produces predictions for the held-back group and prints four measures of how well they match what actually happened.

The confusion matrix is a table of four counts: customers correctly predicted to stay, customers wrongly flagged as leaving, departures the model missed, and departures it caught. Precision is the share of flagged customers who really did leave, so low precision means money spent on customers who were never going to go. Recall is the share of actual departures the model caught, so low recall means customers lost without warning. Precision and recall move against each other, and the balance between them is a commercial decision rather than a technical one.

ROC AUC summarises in one figure between 0.5 and 1.0 how well the model separates the two groups, without depending on where the cut-off for flagging a customer is placed.

`predict_proba` returns a probability for each customer rather than a yes or no. The probability is kept because Stage Four needs a graded measure of risk, not a label.

In [17]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred  = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, digits=3))
print("ROC AUC:", round(roc_auc_score(y_test, y_proba), 3))

[[932 362]
 [ 97 370]]

              precision    recall  f1-score   support

           0      0.906     0.720     0.802      1294
           1      0.505     0.792     0.617       467

    accuracy                          0.739      1761
   macro avg      0.706     0.756     0.710      1761
weighted avg      0.800     0.739     0.753      1761

ROC AUC: 0.84


## 6. Examine the coefficients

This prints the coefficient attached to each measure, sorted from highest to lowest.

The coefficients are the reason for choosing logistic regression: the model's reasoning can be read directly. A positive coefficient means the measure raises the chance of leaving, and a negative one means it lowers it. Because every measure was put on a common scale in the previous step, the sizes can be compared against each other.

In [18]:
coefs = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": logreg.named_steps["model"].coef_[0]
}).sort_values("coefficient", ascending=False)

coefs

,feature,coefficient
1,monthly_charges,1.835196
5,contract_risk,0.733298
2,total_charges,0.573623
8,internet_service_No,0.263224
6,automatic_payment,-0.172772
7,internet_service_Fiber optic,-0.269150
3,has_phone,-0.594036
4,addon_count,-0.821713
0,tenure_months,-1.218183


## 7. Remove the duplicated measure and fit again

This drops `total_charges` from the list of measures, trains the model again on what remains, and prints the new ROC AUC alongside the new coefficients.

`total_charges` is close to `monthly_charges` multiplied by `tenure_months`, so it carries little that the other two do not already supply. When two measures hold overlapping information, the model splits the effect between them in a way that is not stable, and the coefficients stop being reliable even though the predictions remain sound. Since the coefficients are the whole reason this model was chosen, that matters.

Comparing the ROC AUC before and after removal is what shows whether anything was lost by taking it out.

In [19]:
features_v2 = [f for f in X.columns if f != "total_charges"]

X_train2, X_test2 = X_train[features_v2], X_test[features_v2]

logreg2 = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
logreg2.fit(X_train2, y_train)

print("ROC AUC:", round(roc_auc_score(y_test, logreg2.predict_proba(X_test2)[:,1]), 3))

pd.DataFrame({
    "feature": X_train2.columns,
    "coefficient": logreg2.named_steps["model"].coef_[0]
}).sort_values("coefficient", ascending=False)


ROC AUC: 0.838


,feature,coefficient
1,monthly_charges,2.027357
4,contract_risk,0.727646
7,internet_service_No,0.328214
5,automatic_payment,-0.180307
6,internet_service_Fiber optic,-0.275306
2,has_phone,-0.588961
0,tenure_months,-0.742260
3,addon_count,-0.797284


## 8. Check the fibre coefficient against the raw rates

This queries the departure rate for each type of internet service straight from the database.

A coefficient always describes an effect with every other measure held still, which makes it easy to misread on its own. Putting the coefficient next to the plain departure rates for each service type is what settles whether the model is saying something odd or something the data supports.

In [20]:
con.execute("""
    SELECT internet_service, COUNT(*) AS customers, ROUND(AVG(churn_value),3) AS churn_rate
    FROM vw_customer_features GROUP BY 1 ORDER BY 3 DESC
""").df()

,internet_service,customers,churn_rate
0,Fiber optic,3096,0.419
1,DSL,2421,0.190
2,No,1526,0.074


## 9. Build the gradient boosting model

This trains a second model of a different kind on the same measures and prints the same performance figures, so the two can be compared.

Rather than fitting one equation, gradient boosting builds a large number of small decision rules in sequence, each correcting the mistakes of the ones before it. That lets it pick up patterns a single equation cannot, such as high charges mattering only for customers on month-to-month contracts.

No scaling is applied. Models built from decision rules split the data at thresholds rather than measuring distances between values, so the units a measure is recorded in do not affect them.

In [21]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = HistGradientBoostingClassifier(random_state=42)
gb.fit(X_train2, y_train)

y_pred_gb  = gb.predict(X_test2)
y_proba_gb = gb.predict_proba(X_test2)[:, 1]

print(classification_report(y_test, y_pred_gb, digits=3))
print("ROC AUC:", round(roc_auc_score(y_test, y_proba_gb), 3))

              precision    recall  f1-score   support

           0      0.828     0.894     0.860      1294
           1      0.624     0.486     0.546       467

    accuracy                          0.786      1761
   macro avg      0.726     0.690     0.703      1761
weighted avg      0.774     0.786     0.777      1761

ROC AUC: 0.829


## 10. Repeat the comparison under the same conditions

The first comparison was not run on equal terms. The logistic regression was trained with `class_weight="balanced"` and the gradient boosting model was not, so the two were aiming at different things.

This trains the gradient boosting model again with the same class weighting applied, and prints the same figures. Only then do the two sets of numbers describe the same task. Both runs are kept in the notebook rather than the first being deleted.

In [22]:
gb = HistGradientBoostingClassifier(random_state=42, class_weight="balanced")
gb.fit(X_train2, y_train)

y_pred_gb  = gb.predict(X_test2)
y_proba_gb = gb.predict_proba(X_test2)[:, 1]

print(classification_report(y_test, y_pred_gb, digits=3))
print("ROC AUC:", round(roc_auc_score(y_test, y_proba_gb), 3))

              precision    recall  f1-score   support

           0      0.895     0.733     0.806      1294
           1      0.507     0.762     0.609       467

    accuracy                          0.740      1761
   macro avg      0.701     0.747     0.707      1761
weighted avg      0.792     0.740     0.754      1761

ROC AUC: 0.833


---

## 11. Score every customer

The measurement above used only the held-back customers. Stage Four needs a probability for all 7,043, so that value can be added up across the whole customer base.

This applies the chosen model to every customer and puts the resulting probability next to their customer value and value group. It also records which of the two groups from section 3 each customer was placed in, so that the held-back customers can be identified again later.

One point has to be stated plainly. Most of these customers were used to train the model, so their probabilities are slightly flattering, the model having already seen them. That is acceptable here because the purpose of the step is to show how the decision to contact is made, not to measure accuracy. Every performance figure in this notebook comes from the held-back group, which stays untouched.

In [23]:
X_all = X[features_v2]

scored = df[["customer_id", "clv", "value_decile", "churn_value"]].copy()
scored["churn_probability"] = logreg2.predict_proba(X_all)[:, 1]

# Which of the two groups each customer was placed in at section 3. The
# reporting layer needs this to measure the model on the held-back customers
# only, rather than on customers it was trained on.
scored["data_split"] = ["test" if t else "train" for t in X.index.isin(X_test.index)]

print(scored.shape)
print(scored["data_split"].value_counts())
scored.head()


(7043, 6)
data_split
train    5282
test     1761
Name: count, dtype: int64


,customer_id,clv,value_decile,churn_value,churn_probability,data_split
0,7569-NMZYQ,1282.50,1,0,0.127289,train
1,8984-HPEMB,1281.42,1,0,0.176850,train
2,5989-AXPUC,1280.88,1,0,0.189923,train
3,9924-JPRMC,1276.56,1,0,0.168200,test
4,3810-DVDQQ,1270.08,1,0,0.118914,test


## 12. Write the scores to the database

This saves the scored customers as a table so that the economics notebook can read them without training anything again.

It is stored as a table rather than a view, and it is the one place in the project where storing a result is the right choice. Every other layer recalculates from source each time it runs. Model scores cannot, because they come from a trained model that exists only while this notebook is running.

In [24]:
con.execute("DROP TABLE IF EXISTS customer_scores")
con.execute("CREATE TABLE customer_scores AS SELECT * FROM scored")

con.execute("SELECT COUNT(*) FROM customer_scores").df()

,count_star()
0,7043


---

Stage Three ends here. The findings, including the performance of the chosen model, the removal of the duplicated measure, the reading of the fibre coefficient and the comparison between the two models, are set out in `md/03_modelling_writeup.md`.

**Next stage:** combine these probabilities with the customer values worked out in SQL, and decide which customers are worth the cost of contacting.